In [5]:
# verifying Runtime
import torch
import os

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Do not start model training yet.")


PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [6]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/GLYRA"

# Reload validation CSV
path = os.path.join(DATA_DIR, "Development_Validation_SequenceReady.csv")
df = pd.read_csv(path)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Reload existing validation sequences
npz_path = os.path.join(DATA_DIR, "Development_Validation_Sequences.npz")

data = np.load(npz_path)
X_val = data["X"]
Y_val = data["Y"]

print("Validation CSV:", df.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)
print("\nDataset counts:")
print(df["Dataset"].value_counts())


Validation CSV: (1004459, 34)
X_val: (934004, 8, 27)
Y_val: (934004, 4)

Dataset counts:
Dataset
REPLACE-BG       979908
Shanghai-T2DM     24551
Name: count, dtype: int64


In [ ]:
import os

DRIVE_FOLDER = "/content/drive/MyDrive/GLYRA"

required_files = [
    "Development_Train_Sequences.npz",
    "Development_Validation_Sequences.npz",
    "normalization_parameters.json"
]

for filename in required_files:
    path = os.path.join(DRIVE_FOLDER, filename)
    status = "FOUND" if os.path.isfile(path) else "MISSING"
    print(f"{filename}: {status}")

Development_Train_Sequences.npz: FOUND
Development_Validation_Sequences.npz: FOUND
normalization_parameters.json: FOUND


In [8]:

import os
import json
import numpy as np

DRIVE_FOLDER = "/content/drive/MyDrive/GLYRA"

train_path = os.path.join(
    DRIVE_FOLDER, "Development_Train_Sequences.npz"
)
val_path = os.path.join(
    DRIVE_FOLDER, "Development_Validation_Sequences.npz"
)
norm_path = os.path.join(
    DRIVE_FOLDER, "normalization_parameters.json"
)

# Load sequences into CPU RAM
with np.load(train_path) as data:
    X_train = data["X"]
    Y_train = data["Y"]

with np.load(val_path) as data:
    X_val = data["X"]
    Y_val = data["Y"]

# Load training normalization parameters
with open(norm_path, "r") as f:
    normalization_params = json.load(f)

print("Training input:", X_train.shape)
print("Training targets:", Y_train.shape)
print("Validation input:", X_val.shape)
print("Validation targets:", Y_val.shape)

print("\nAll arrays are finite:",
      all(np.isfinite(a).all()
          for a in [X_train, Y_train, X_val, Y_val]))

print("Normalization parameters loaded:",
      len(normalization_params))


Training input: (3816404, 8, 27)
Training targets: (3816404, 4)
Validation input: (934004, 8, 27)
Validation targets: (934004, 4)

All arrays are finite: True
Normalization parameters loaded: 11


In [9]:
# Create a Model

import torch
import torch.nn as nn

# Use the GPU when available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class GlyraGPFormer(nn.Module):
    """
    Initial proposed Glyra model:
    shared temporal encoder with feature-availability-based domain guidance.
    Input:  (batch, 8, 27)
    Output: (batch, 4)
    """

    def __init__(
        self,
        input_dim=27,
        history_length=8,
        hidden_dim=64,
        num_heads=4,
        num_layers=2,
        dropout=0.1,
        output_dim=4
    ):
        super().__init__()

        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")

        self.input_dim = input_dim
        self.history_length = history_length

        # 1. Project input features into a hidden representation
        self.feature_projection = nn.Linear(input_dim, hidden_dim)

        # 2. Learnable temporal position embeddings
        self.temporal_embedding = nn.Parameter(
            torch.zeros(1, history_length, hidden_dim)
        )
        nn.init.normal_(self.temporal_embedding, mean=0.0, std=0.02)

        # 3. Domain guidance from the seven dataset-level
        # feature-availability masks (feature indices 13 to 19)
        self.domain_projection = nn.Sequential(
            nn.Linear(7, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 4. Shared temporal Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.temporal_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.final_norm = nn.LayerNorm(hidden_dim)

        # 5. Multi-horizon glucose prediction head
        self.forecast_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        # Expected x: (batch, 8, 27)
        if x.ndim != 3:
            raise ValueError("Input must have shape (batch, 8, 27)")

        if x.shape[1] != self.history_length:
            raise ValueError("Unexpected history length")

        if x.shape[2] != self.input_dim:
            raise ValueError("Unexpected number of input features")

        # Dataset-level availability masks: indices 13 through 19
        availability = x[:, :, 13:20].mean(dim=1)

        # Feature embedding + temporal positions
        h = self.feature_projection(x)
        h = h + self.temporal_embedding

        # Add the same domain-guidance vector to each time point
        domain_context = self.domain_projection(availability)
        h = h + domain_context.unsqueeze(1)

        # Temporal encoding
        h = self.temporal_encoder(h)
        h = self.final_norm(h)

        # Use the final historical position for forecasting
        context = h[:, -1, :]

        # Four normalized glucose predictions
        predictions = self.forecast_head(context)

        return predictions


# Instantiate the model
model = GlyraGPFormer().to(DEVICE)

print("Device:", DEVICE)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Model created successfully.")


Device: cuda
Model parameters: 111492
Model created successfully.


In [10]:
import torch
import os

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/GLYRA/checkpoints/GPFormer_best.pth"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Checkpoint loaded successfully!")
print("Model:", "GlyraGPFormer")
print("Best epoch:", checkpoint["epoch"])
print("Best validation MSE:", checkpoint["val_mse"])

Checkpoint loaded successfully!
Model: GlyraGPFormer
Best epoch: 8
Best validation MSE: 0.33379001181389156


In [ ]:

# Use a small batch from the real training data
X_test_batch = torch.from_numpy(X_train[:4]).to(DEVICE)

model.eval()

with torch.no_grad():
    predictions = model(X_test_batch)

print("Input shape:", tuple(X_test_batch.shape))
print("Output shape:", tuple(predictions.shape))
print("Predictions finite:", torch.isfinite(predictions).all().item())
print("Predictions (normalized):")
print(predictions.cpu().numpy())


Input shape: (4, 8, 27)
Output shape: (4, 4)
Predictions finite: True
Predictions (normalized):
[[ 0.02968945 -0.16396771 -0.27851778 -0.12297507]
 [ 0.08502116 -0.10722964 -0.19652206 -0.06079401]
 [ 0.09030699 -0.10840414 -0.18789591 -0.07331991]
 [ 0.09107929 -0.11546935 -0.1802026  -0.08104108]]


In [11]:
from torch.utils.data import TensorDataset, DataLoader

# Convert NumPy arrays to PyTorch tensors without copying the full arrays
X_train_tensor = torch.from_numpy(X_train)
Y_train_tensor = torch.from_numpy(Y_train)

X_val_tensor = torch.from_numpy(X_val)
Y_val_tensor = torch.from_numpy(Y_val)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

# Create mini-batch loaders
BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

# Test one batch
X_batch, Y_batch = next(iter(train_loader))

print("Input batch shape:", tuple(X_batch.shape))
print("Target batch shape:", tuple(Y_batch.shape))

Training samples: 3816404
Validation samples: 934004
Training batches: 7454
Validation batches: 1825
Input batch shape: (512, 8, 27)
Target batch shape: (512, 4)


In [12]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# Create validation loader if not already available
X_val_tensor = torch.from_numpy(X_val)
Y_val_tensor = torch.from_numpy(Y_val)

val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Evaluate trained model
model.eval()

total_squared_error = 0.0
total_elements = 0

with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        Y_batch = Y_batch.to(DEVICE, non_blocking=True)

        predictions = model(X_batch)

        squared_error = torch.sum((predictions - Y_batch) ** 2)

        total_squared_error += squared_error.item()
        total_elements += Y_batch.numel()

validation_mse = total_squared_error / total_elements

print("Checkpoint evaluation completed!")
print("Validation MSE:", validation_mse)
print("Validation samples:", len(val_dataset))

Checkpoint evaluation completed!
Validation MSE: 0.3337900118135725
Validation samples: 934004


In [ ]:

import os
import torch

# Create a folder for model checkpoints
CHECKPOINT_DIR = "/content/drive/MyDrive/GLYRA/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# File path
checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "GPFormer_checkpoint_after_test.pth"
)

# Save model + optimizer + training information
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "training_batches_completed": 100,
    "model_name": "GlyraGPFormer",
    "batch_size": BATCH_SIZE,
}, checkpoint_path)

print("Checkpoint saved successfully!")
print("Location:", checkpoint_path)
print("File size (MB):", round(os.path.getsize(checkpoint_path) / (1024**2), 2))


Checkpoint saved successfully!
Location: /content/drive/MyDrive/GLYRA/checkpoints/GPFormer_checkpoint_after_test.pth
File size (MB): 1.32


In [ ]:

import os

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print("Checkpoint size (MB):",
          round(os.path.getsize(checkpoint_path) / (1024**2), 2))


Checkpoint exists: True
Checkpoint size (MB): 1.32


In [ ]:

import os
import torch
import torch.nn as nn
from torch.optim import AdamW

# Training configuration
EPOCHS = 10
PATIENCE = 3
MAX_GRAD_NORM = 1.0

# Save full-training checkpoints separately from the test checkpoint
CHECKPOINT_DIR = "/content/drive/MyDrive/GLYRA/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR, "GPFormer_best.pth"
)
LATEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR, "GPFormer_latest.pth"
)

# Use the existing model and optimizer
criterion = nn.MSELoss()

best_val_loss = float("inf")
epochs_without_improvement = 0
history = []

for epoch in range(EPOCHS):
    # ---------------- TRAINING ----------------
    model.train()
    train_loss_sum = 0.0
    train_samples_seen = 0

    for batch_idx, (X_batch, Y_batch) in enumerate(train_loader):
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        Y_batch = Y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        predictions = model(X_batch)
        loss = criterion(predictions, Y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), MAX_GRAD_NORM
        )
        optimizer.step()

        batch_size = X_batch.size(0)
        train_loss_sum += loss.item() * batch_size
        train_samples_seen += batch_size

        if (batch_idx + 1) % 500 == 0:
            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.5f}"
            )

    avg_train_loss = train_loss_sum / train_samples_seen

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss_sum = 0.0
    val_samples_seen = 0

    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            Y_batch = Y_batch.to(DEVICE, non_blocking=True)

            predictions = model(X_batch)
            loss = criterion(predictions, Y_batch)

            batch_size = X_batch.size(0)
            val_loss_sum += loss.item() * batch_size
            val_samples_seen += batch_size

    avg_val_loss = val_loss_sum / val_samples_seen

    history.append({
        "epoch": epoch + 1,
        "train_mse": avg_train_loss,
        "val_mse": avg_val_loss
    })

    print(f"\nEpoch {epoch + 1}/{EPOCHS} completed")
    print(f"Training MSE:   {avg_train_loss:.6f}")
    print(f"Validation MSE: {avg_val_loss:.6f}")

    # Save the latest recovery checkpoint every epoch
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_mse": avg_train_loss,
        "val_mse": avg_val_loss,
        "history": history,
        "batch_size": BATCH_SIZE
    }
    torch.save(checkpoint, LATEST_CHECKPOINT_PATH)

    # Save the best model based on validation MSE
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        torch.save(checkpoint, BEST_CHECKPOINT_PATH)
        print("Best checkpoint saved.")
    else:
        epochs_without_improvement += 1
        print(
            "No validation improvement:",
            epochs_without_improvement,
            "epoch(s)"
        )

    print("-" * 50)

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining finished.")
print(f"Best validation MSE: {best_val_loss:.6f}")
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Latest checkpoint:", LATEST_CHECKPOINT_PATH)


Epoch 1/10 | Batch 500/7454 | Loss: 0.36165
Epoch 1/10 | Batch 1000/7454 | Loss: 0.35714
Epoch 1/10 | Batch 1500/7454 | Loss: 0.38908
Epoch 1/10 | Batch 2000/7454 | Loss: 0.36095
Epoch 1/10 | Batch 2500/7454 | Loss: 0.36782
Epoch 1/10 | Batch 3000/7454 | Loss: 0.41065
Epoch 1/10 | Batch 3500/7454 | Loss: 0.33221
Epoch 1/10 | Batch 4000/7454 | Loss: 0.35445
Epoch 1/10 | Batch 4500/7454 | Loss: 0.41344
Epoch 1/10 | Batch 5000/7454 | Loss: 0.34737
Epoch 1/10 | Batch 5500/7454 | Loss: 0.36631
Epoch 1/10 | Batch 6000/7454 | Loss: 0.37941
Epoch 1/10 | Batch 6500/7454 | Loss: 0.36317
Epoch 1/10 | Batch 7000/7454 | Loss: 0.30225

Epoch 1/10 completed
Training MSE:   0.358075
Validation MSE: 0.343582
Best checkpoint saved.
--------------------------------------------------
Epoch 2/10 | Batch 500/7454 | Loss: 0.38857
Epoch 2/10 | Batch 1000/7454 | Loss: 0.38602
Epoch 2/10 | Batch 1500/7454 | Loss: 0.41128
Epoch 2/10 | Batch 2000/7454 | Loss: 0.32215
Epoch 2/10 | Batch 2500/7454 | Loss: 0.34084
E

In [ ]:

import os
import torch

CHECKPOINT_DIR = "/content/drive/MyDrive/GLYRA/checkpoints"
BEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR, "GPFormer_best.pth"
)

print("Best checkpoint exists:",
      os.path.exists(BEST_CHECKPOINT_PATH))

print("Checkpoint files:",
      sorted(os.listdir(CHECKPOINT_DIR)))

if os.path.exists(BEST_CHECKPOINT_PATH):
    checkpoint = torch.load(
        BEST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=True
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print("Best model loaded successfully.")
    print("Best epoch:", checkpoint["epoch"])
    print("Saved validation MSE:", checkpoint["val_mse"])


Best checkpoint exists: True
Checkpoint files: ['GPFormer_best.pth', 'GPFormer_checkpoint_after_test.pth', 'GPFormer_latest.pth']
Best model loaded successfully.
Best epoch: 8
Saved validation MSE: 0.33379001181389156


In [ ]:

import json

NORMALIZATION_PATH = (
    "/content/drive/MyDrive/GLYRA/normalization_parameters.json"
)

with open(NORMALIZATION_PATH, "r") as f:
    norm_params = json.load(f)

print("Normalization parameter structure:")
print(json.dumps(norm_params, indent=2)[:5000])


Normalization parameter structure:
{
  "CGM": {
    "mean": 159.79780684269394,
    "std": 64.19917584332617
  },
  "Carbs": {
    "mean": 1.2142986611935618,
    "std": 8.26936492082255
  },
  "Meal_Mass_g": {
    "mean": 280.2185514612452,
    "std": 170.77980697159947
  },
  "Bolus_Insulin": {
    "mean": 0.22023264311812873,
    "std": 1.1774110468213836
  },
  "Basal_Insulin": {
    "mean": 0.23165614561752834,
    "std": 0.15800246563450573
  },
  "CBG": {
    "mean": 160.468085106383,
    "std": 56.83233329733735
  },
  "Blood_Ketone": {
    "mean": 2.937777777777778,
    "std": 5.5469220694403525
  },
  "Target_30": {
    "mean": 159.71675250558377,
    "std": 64.09157593369515
  },
  "Target_60": {
    "mean": 159.67699310734878,
    "std": 64.03934349380235
  },
  "Target_90": {
    "mean": 159.64498855453516,
    "std": 64.00913540737415
  },
  "Target_120": {
    "mean": 159.6306046049541,
    "std": 63.99042789019152
  }
}


In [ ]:

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Forecast target names and their normalization parameters
target_names = ["Target_30", "Target_60", "Target_90", "Target_120"]
horizons = ["30 min", "60 min", "90 min", "120 min"]

target_means = np.array(
    [norm_params[name]["mean"] for name in target_names],
    dtype=np.float64
)
target_stds = np.array(
    [norm_params[name]["std"] for name in target_names],
    dtype=np.float64
)

# Load the best checkpoint (safe to load again)
checkpoint = torch.load(
    "/content/drive/MyDrive/GLYRA/checkpoints/GPFormer_best.pth",
    map_location=DEVICE,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Evaluate all validation sequences
all_predictions = []
all_targets = []

with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)

        predictions = model(X_batch)

        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(Y_batch.numpy())

pred_norm = np.concatenate(all_predictions, axis=0)
true_norm = np.concatenate(all_targets, axis=0)

# Convert normalized values back to mg/dL using each target's own statistics
pred_mgdl = pred_norm * target_stds + target_means
true_mgdl = true_norm * target_stds + target_means

# Calculate metrics separately for each horizon
results = []

for i, horizon in enumerate(horizons):
    mae = mean_absolute_error(true_mgdl[:, i], pred_mgdl[:, i])
    rmse = np.sqrt(mean_squared_error(true_mgdl[:, i], pred_mgdl[:, i]))
    r2 = r2_score(true_mgdl[:, i], pred_mgdl[:, i])

    results.append({
        "Horizon": horizon,
        "MAE (mg/dL)": mae,
        "RMSE (mg/dL)": rmse,
        "R²": r2
    })

results_df = pd.DataFrame(results)

print("Validation sequences evaluated:", len(true_mgdl))
print("Best checkpoint epoch:", checkpoint["epoch"])
print("\nValidation metrics in original glucose units:")
display(results_df.round(3))

print("\nAll predictions finite:", np.isfinite(pred_mgdl).all())
print("All targets finite:", np.isfinite(true_mgdl).all())


Validation sequences evaluated: 934004
Best checkpoint epoch: 8

Validation metrics in original glucose units:


,Horizon,MAE (mg/dL),RMSE (mg/dL),R²
0,30 min,12.775,18.358,0.919
1,60 min,23.077,32.180,0.751
2,90 min,30.774,41.816,0.580
3,120 min,36.326,48.473,0.436



All predictions finite: True
All targets finite: True


In [ ]:

import os

DATA_DIR = "/content/drive/MyDrive/GLYRA"

for filename in [
    "Development_Validation_SequenceReady.csv",
    "Development_Validation.csv",
    "REPLACE_BG_15min_prepared.csv",
    "Shanghai_T2DM_15min_prepared.csv"
]:
    path = os.path.join(DATA_DIR, filename)
    print(f"{filename}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")


In [ ]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/GLYRA"

path = os.path.join(DATA_DIR, "Development_Validation_SequenceReady.csv")

df_check = pd.read_csv(path, nrows=5)

print("Columns:")
print(df_check.columns.tolist())

print("\nFirst 5 rows:")
display(df_check)

print("\nDataset column available:", "Dataset" in df_check.columns)


In [ ]:

import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/GLYRA"

path = os.path.join(DATA_DIR, "Development_Validation_SequenceReady.csv")
df = pd.read_csv(path)

print("Rows by dataset:")
print(df.groupby("Dataset").size())

print("\nUnique patients by dataset:")
print(df.groupby("Dataset")["PtID"].nunique())

target_cols = ["Target_30", "Target_60", "Target_90", "Target_120"]

print("\nRows with all four targets available:")
print(
    df.groupby("Dataset")[target_cols]
      .apply(lambda group: group.notna().all(axis=1).sum())
)

print("\nRows with missing targets:")
print(df.groupby("Dataset")[target_cols].apply(lambda group: group.isna().sum()))

print("\nTotal rows:", len(df))
print("Expected validation sequences:", len(Y_val))


In [ ]:

import numpy as np
import pandas as pd
import os

DATA_DIR = "/content/drive/MyDrive/GLYRA"
path = os.path.join(DATA_DIR, "Development_Validation_SequenceReady.csv")

df = pd.read_csv(path)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

target_cols = ["Target_30", "Target_60", "Target_90", "Target_120"]

candidate_labels = []
candidate_targets = []

# Reconstruct 8-step histories within each patient and dataset.
for (dataset, patient), group in df.groupby(
    ["Dataset", "PtID"], sort=True
):
    group = group.sort_values("Timestamp").reset_index(drop=True)

    timestamps = group["Timestamp"]
    targets = group[target_cols].to_numpy(dtype=np.float32)

    for end in range(7, len(group)):
        start = end - 7

        # All 8 history points must be exactly 15 minutes apart.
        window_times = timestamps.iloc[start:end + 1]
        gaps = window_times.diff().dropna()

        if not (gaps == pd.Timedelta(minutes=15)).all():
            continue

        # Targets must all be available at the final history point.
        if not np.isfinite(targets[end]).all():
            continue

        candidate_labels.append(dataset)
        candidate_targets.append(targets[end])

candidate_labels = np.asarray(candidate_labels)
candidate_targets = np.asarray(candidate_targets, dtype=np.float32)

print("Reconstructed candidate sequences:", len(candidate_targets))
print("Existing validation sequences:", len(Y_val))
print("Candidate target shape:", candidate_targets.shape)
print("Existing Y_val shape:", Y_val.shape)

if candidate_targets.shape == Y_val.shape:
    exact_match = np.allclose(
        candidate_targets, Y_val, rtol=0, atol=1e-6
    )
    print("Targets match existing sequence order:", exact_match)

    if exact_match:
        print("\nVerified sequence counts by dataset:")
        for dataset in np.unique(candidate_labels):
            print(f"{dataset}: {(candidate_labels == dataset).sum()}")
    else:
        print(
            "\nSTOP: target order or sequence-generation rules differ. "
            "Do not calculate dataset-specific metrics yet."
        )
else:
    print(
        "\nSTOP: sequence counts differ. "
        "Do not calculate dataset-specific metrics yet."
    )


In [ ]:

import numpy as np
import pandas as pd

target_cols = ["Target_30", "Target_60", "Target_90", "Target_120"]

def reconstruct_targets(group_order, sort_groups):
    labels = []
    targets = []

    for keys, group in df.groupby(group_order, sort=sort_groups):
        group = group.sort_values("Timestamp").reset_index(drop=True)
        times = group["Timestamp"]
        values = group[target_cols].to_numpy(dtype=np.float32)

        for end in range(7, len(group)):
            start = end - 7

            if not (
                times.iloc[start:end + 1]
                .diff().dropna()
                .eq(pd.Timedelta(minutes=15)).all()
            ):
                continue

            if not np.isfinite(values[end]).all():
                continue

            # Dataset is the first key when grouping by Dataset, PtID.
            dataset = (
                keys[0] if isinstance(keys, tuple)
                and group_order[0] == "Dataset"
                else group["Dataset"].iloc[end]
            )
            labels.append(dataset)
            targets.append(values[end])

    return np.asarray(labels), np.asarray(targets, dtype=np.float32)

tests = [
    (["Dataset", "PtID"], True),
    (["Dataset", "PtID"], False),
    (["PtID", "Dataset"], True),
    (["PtID", "Dataset"], False),
]

for group_order, sort_groups in tests:
    labels_try, targets_try = reconstruct_targets(
        group_order, sort_groups
    )

    matches = (
        targets_try.shape == Y_val.shape
        and np.allclose(targets_try, Y_val, rtol=0, atol=1e-6)
    )

    print(
        f"group_by={group_order}, sort={sort_groups}: "
        f"count={len(targets_try)}, exact_match={matches}"
    )

    if matches:
        print("Verified dataset counts:")
        for name, count in zip(*np.unique(labels_try, return_counts=True)):
            print(f"  {name}: {count}")


In [ ]:
print("Y_val exists:", 'Y_val' in globals())
print("NumPy available:", 'np' in globals())

In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/GLYRA'
print(os.listdir(DRIVE_DIR))

In [ ]:
# !pip install -q <any package not preinstalled>
import torch, numpy as np, pandas as pd
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)   # should print 'cuda'

In [ ]:
import shutil
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

for f in ['Development_Train_Sequences.npz',
          'Development_Validation_Sequences.npz']:
    dst = f'{LOCAL_DIR}/{f}'
    if not os.path.exists(dst):
        shutil.copy(f'{DRIVE_DIR}/{f}', dst)

In [ ]:
train = np.load(f'{LOCAL_DIR}/Development_Train_Sequences.npz')
val   = np.load(f'{LOCAL_DIR}/Development_Validation_Sequences.npz')
print(train.files)

In [ ]:
CKPT = f'{DRIVE_DIR}/GPFormer_latest.pth'

def save_ckpt(epoch, model, optimizer, scheduler=None, best_val=None):
    tmp = CKPT + '.tmp'
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'best_val': best_val,
    }, tmp)
    os.replace(tmp, CKPT)

In [ ]:
start_epoch, best_val = 0, float('inf')
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    if scheduler and ck.get('scheduler'):
        scheduler.load_state_dict(ck['scheduler'])
    start_epoch = ck['epoch'] + 1
    best_val = ck['best_val']
    print(f'Resumed from epoch {start_epoch}')
else:
    print('Starting fresh')

In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

Device: cuda
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/GLYRA/checkpoints"

print(os.listdir(CHECKPOINT_DIR))

['GPFormer_checkpoint_after_test.pth', 'GPFormer_best.pth', 'GPFormer_latest.pth']


In [4]:
import torch
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/GLYRA/checkpoints"

files = [
    "GPFormer_best.pth",
    "GPFormer_latest.pth",
    "GPFormer_checkpoint_after_test.pth"
]

for filename in files:
    path = os.path.join(CHECKPOINT_DIR, filename)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("\n" + "=" * 45)
    print("FILE:", filename)
    print("Saved information:")

    if isinstance(checkpoint, dict):
        for key, value in checkpoint.items():
            if key == "model_state_dict":
                print("model_state_dict: Available")
            elif key == "optimizer_state_dict":
                print("optimizer_state_dict: Available")
            elif key == "history":
                print("history:", len(value), "epochs recorded")
            elif isinstance(value, (int, float, str)):
                print(f"{key}: {value}")
            else:
                print(f"{key}: {type(value).__name__}")
    else:
        print("Checkpoint type:", type(checkpoint))


FILE: GPFormer_best.pth
Saved information:
epoch: 8
model_state_dict: Available
optimizer_state_dict: Available
train_mse: 0.3470043403224298
val_mse: 0.33379001181389156
history: 8 epochs recorded
batch_size: 512

FILE: GPFormer_latest.pth
Saved information:
epoch: 10
model_state_dict: Available
optimizer_state_dict: Available
train_mse: 0.34605702729507437
val_mse: 0.33554665656921046
history: 10 epochs recorded
batch_size: 512

FILE: GPFormer_checkpoint_after_test.pth
Saved information:
model_state_dict: Available
optimizer_state_dict: Available
training_batches_completed: 100
model_name: GlyraGPFormer
batch_size: 512


In [13]:

import torch
import torch.nn as nn
from torch.optim import AdamW

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

# Short training test
MAX_TRAIN_BATCHES = 100
MAX_VAL_BATCHES = 50

model.train()
train_loss_sum = 0.0
train_batches_done = 0

for batch_idx, (X_batch, Y_batch) in enumerate(train_loader):
    if batch_idx >= MAX_TRAIN_BATCHES:
        break

    X_batch = X_batch.to(DEVICE, non_blocking=True)
    Y_batch = Y_batch.to(DEVICE, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)

    predictions = model(X_batch)
    loss = criterion(predictions, Y_batch)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_loss_sum += loss.item()
    train_batches_done += 1

    if (batch_idx + 1) % 20 == 0:
        print(
            f"Training batch {batch_idx + 1}/{MAX_TRAIN_BATCHES} "
            f"| Loss: {loss.item():.6f}"
        )

avg_train_loss = train_loss_sum / train_batches_done

# Short validation test: no parameter updates
model.eval()
val_loss_sum = 0.0
val_batches_done = 0

with torch.no_grad():
    for batch_idx, (X_batch, Y_batch) in enumerate(val_loader):
        if batch_idx >= MAX_VAL_BATCHES:
            break

        X_batch = X_batch.to(DEVICE, non_blocking=True)
        Y_batch = Y_batch.to(DEVICE, non_blocking=True)

        predictions = model(X_batch)
        loss = criterion(predictions, Y_batch)

        val_loss_sum += loss.item()
        val_batches_done += 1

avg_val_loss = val_loss_sum / val_batches_done

print("\nShort training test completed.")
print(f"Average training MSE:   {avg_train_loss:.6f}")
print(f"Average validation MSE: {avg_val_loss:.6f}")
print("Model output finite:", torch.isfinite(predictions).all().item())

Training batch 20/100 | Loss: 0.350444
Training batch 40/100 | Loss: 0.324532
Training batch 60/100 | Loss: 0.340821
Training batch 80/100 | Loss: 0.368040
Training batch 100/100 | Loss: 0.358660

Short training test completed.
Average training MSE:   0.348128
Average validation MSE: 0.151997
Model output finite: True


In [14]:

import os
import torch

checkpoint_dir = "/content/drive/MyDrive/GLYRA/checkpoints"

for filename in os.listdir(checkpoint_dir):
    if filename.endswith(".pth"):
        path = os.path.join(checkpoint_dir, filename)

        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

        print("\nCheckpoint:", filename)
        print("Saved epoch:", checkpoint.get("epoch", "Not recorded"))
        print("Best validation loss:",
              checkpoint.get("best_val_loss", "Not recorded"))



Checkpoint: GPFormer_checkpoint_after_test.pth
Saved epoch: Not recorded
Best validation loss: Not recorded

Checkpoint: GPFormer_best.pth
Saved epoch: 8
Best validation loss: Not recorded

Checkpoint: GPFormer_latest.pth
Saved epoch: 10
Best validation loss: Not recorded


In [16]:
import torch
import os

folder = "/content/drive/MyDrive/GLYRA/checkpoints"

for name in ["GPFormer_best.pth", "GPFormer_latest.pth"]:
    ckpt = torch.load(
        os.path.join(folder, name),
        map_location="cpu",
        weights_only=True
    )

    print("\n", name)
    print("Keys:", list(ckpt.keys()))

    for key, value in ckpt.items():
        if "loss" in key.lower() or "epoch" in key.lower():
            print(key, "=", value)


 GPFormer_best.pth
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_mse', 'val_mse', 'history', 'batch_size']
epoch = 8

 GPFormer_latest.pth
Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_mse', 'val_mse', 'history', 'batch_size']
epoch = 10


In [17]:
import torch
import os

path = "/content/drive/MyDrive/GLYRA/checkpoints/GPFormer_best.pth"

ckpt = torch.load(path, map_location="cpu", weights_only=True)

print("Best checkpoint epoch:", ckpt["epoch"])
print("Training MSE:", ckpt["train_mse"])
print("Validation MSE:", ckpt["val_mse"])
print("Batch size:", ckpt["batch_size"])

print("\nTraining history:")
history = ckpt["history"]
print("History type:", type(history).__name__)

if isinstance(history, dict):
    for key, values in history.items():
        print(key, ":", values)
elif isinstance(history, (list, tuple)):
    for item in history:
        print(item)
else:
    print(history)

Best checkpoint epoch: 8
Training MSE: 0.3470043403224298
Validation MSE: 0.33379001181389156
Batch size: 512

Training history:
History type: list
{'epoch': 1, 'train_mse': 0.3580745456058731, 'val_mse': 0.34358181441201924}
{'epoch': 2, 'train_mse': 0.3539958109576974, 'val_mse': 0.3395392668114179}
{'epoch': 3, 'train_mse': 0.3517113046210795, 'val_mse': 0.3395684324878981}
{'epoch': 4, 'train_mse': 0.3502293358302783, 'val_mse': 0.33594294158711174}
{'epoch': 5, 'train_mse': 0.3491594058531502, 'val_mse': 0.33577724773715006}
{'epoch': 6, 'train_mse': 0.34833983223994347, 'val_mse': 0.3351427601146424}
{'epoch': 7, 'train_mse': 0.3475383105999752, 'val_mse': 0.3396959120850577}
{'epoch': 8, 'train_mse': 0.3470043403224298, 'val_mse': 0.33379001181389156}


In [18]:
print("Best checkpoint epoch:", ckpt["epoch"])
print("Best validation MSE:", ckpt["val_mse"])
print("Latest checkpoint epoch:", 10)

Best checkpoint epoch: 8
Best validation MSE: 0.33379001181389156
Latest checkpoint epoch: 10


In [19]:
import os

folder = "/content/drive/MyDrive/GLYRA/checkpoints"

for name in os.listdir(folder):
    print(name)

GPFormer_checkpoint_after_test.pth
GPFormer_best.pth
GPFormer_latest.pth


In [20]:
import torch
import os

path = "/content/drive/MyDrive/GLYRA/checkpoints/GPFormer_checkpoint_after_test.pth"

ckpt = torch.load(path, map_location="cpu", weights_only=True)

print("Keys:", list(ckpt.keys()))

for key, value in ckpt.items():
    if key != "model_state_dict" and key != "optimizer_state_dict":
        print(key, ":", value)

Keys: ['model_state_dict', 'optimizer_state_dict', 'training_batches_completed', 'model_name', 'batch_size']
training_batches_completed : 100
model_name : GlyraGPFormer
batch_size : 512


In [21]:
import shutil
import os

source = "/content/drive/MyDrive/GLYRA/checkpoints/GPFormer_best.pth"
backup = "/content/drive/MyDrive/GLYRA/checkpoints/GPFormer_best_backup_epoch8.pth"

shutil.copy2(source, backup)

print("Backup created:", os.path.exists(backup))

Backup created: True


In [22]:
import torch

checkpoint_path = (
    "/content/drive/MyDrive/GLYRA/checkpoints/"
    "GPFormer_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Model loaded successfully")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation MSE:", checkpoint["val_mse"])

Model loaded successfully
Checkpoint epoch: 8
Validation MSE: 0.33379001181389156


In [23]:
import json

path = "/content/drive/MyDrive/GLYRA/normalization_parameters.json"

with open(path, "r") as f:
    norm_params = json.load(f)

for target in ["Target_30", "Target_60", "Target_90", "Target_120"]:
    print(target, ":", norm_params[target])

Target_30 : {'mean': 159.71675250558377, 'std': 64.09157593369515}
Target_60 : {'mean': 159.67699310734878, 'std': 64.03934349380235}
Target_90 : {'mean': 159.64498855453516, 'std': 64.00913540737415}
Target_120 : {'mean': 159.6306046049541, 'std': 63.99042789019152}


In [24]:
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        X_batch = X_batch.to(DEVICE)

        predictions = model(X_batch)

        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(Y_batch.numpy())

predictions = np.concatenate(all_predictions, axis=0)
targets = np.concatenate(all_targets, axis=0)

target_names = ["Target_30", "Target_60", "Target_90", "Target_120"]

print("Validation samples:", len(targets))
print("\nEVALUATION RESULTS (mg/dL)")
print("-" * 65)

for i, name in enumerate(target_names):
    mean = norm_params[name]["mean"]
    std = norm_params[name]["std"]

    actual = targets[:, i] * std + mean
    predicted = predictions[:, i] * std + mean

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)

    print(f"\n{name} ({(i+1)*30} minutes)")
    print(f"MAE  : {mae:.3f} mg/dL")
    print(f"RMSE : {rmse:.3f} mg/dL")
    print(f"R²   : {r2:.4f}")

Validation samples: 934004

EVALUATION RESULTS (mg/dL)
-----------------------------------------------------------------

Target_30 (30 minutes)
MAE  : 12.775 mg/dL
RMSE : 18.358 mg/dL
R²   : 0.9191

Target_60 (60 minutes)
MAE  : 23.077 mg/dL
RMSE : 32.180 mg/dL
R²   : 0.7514

Target_90 (90 minutes)
MAE  : 30.774 mg/dL
RMSE : 41.816 mg/dL
R²   : 0.5803

Target_120 (120 minutes)
MAE  : 36.326 mg/dL
RMSE : 48.473 mg/dL
R²   : 0.4360


In [25]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = np.asarray(X_val)
Y = np.asarray(Y_val)

# Find the most recent observed CGM value in each 2-hour history
cgm_observed = X[:, :, 20] > 0.5
valid_samples = cgm_observed.any(axis=1)

reverse_index = cgm_observed[:, ::-1].argmax(axis=1)
last_index = 7 - reverse_index

rows = np.arange(len(X))
latest_cgm_normalized = X[rows, last_index, 0]

# Convert latest CGM to mg/dL
cgm_mean = norm_params["CGM"]["mean"]
cgm_std = norm_params["CGM"]["std"]

latest_cgm = latest_cgm_normalized * cgm_std + cgm_mean

target_names = ["Target_30", "Target_60", "Target_90", "Target_120"]

print("Total validation sequences:", len(X))
print("Sequences with observed CGM:", valid_samples.sum())

print("\nPERSISTENCE BASELINE vs GPFormer")
print("-" * 65)

for i, name in enumerate(target_names):
    mean = norm_params[name]["mean"]
    std = norm_params[name]["std"]

    actual = Y[:, i] * std + mean
    baseline = latest_cgm

    actual_valid = actual[valid_samples]
    baseline_valid = baseline[valid_samples]
    gpformer_valid = predictions[valid_samples, i] * std + mean

    print(f"\n{name} ({(i+1)*30} minutes)")
    print(
        f"Persistence MAE: "
        f"{mean_absolute_error(actual_valid, baseline_valid):.3f}"
    )
    print(
        f"GPFormer MAE:    "
        f"{mean_absolute_error(actual_valid, gpformer_valid):.3f}"
    )

Total validation sequences: 934004
Sequences with observed CGM: 934004

PERSISTENCE BASELINE vs GPFormer
-----------------------------------------------------------------

Target_30 (30 minutes)
Persistence MAE: 16.589
GPFormer MAE:    12.775

Target_60 (60 minutes)
Persistence MAE: 28.355
GPFormer MAE:    23.077

Target_90 (90 minutes)
Persistence MAE: 37.170
GPFormer MAE:    30.774

Target_120 (120 minutes)
Persistence MAE: 43.919
GPFormer MAE:    36.326


In [26]:
import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/GLYRA/Development_Validation_SequenceReady.csv"

df = pd.read_csv(path)

INPUT_STEPS = 8

TARGET_COLUMNS = [
    "Target_30",
    "Target_60",
    "Target_90",
    "Target_120"
]

df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="mixed",
    errors="coerce"
)

df = df.dropna(subset=["Timestamp"]).reset_index(drop=True)

df["PatientKey"] = (
    df["Dataset"].astype(str)
    + "_"
    + df["PtID"].astype(str)
)

df = df.sort_values(
    ["PatientKey", "Timestamp"]
).reset_index(drop=True)

df["PreviousTimestamp"] = (
    df.groupby("PatientKey")["Timestamp"].shift(1)
)

df["DiffMinutes"] = (
    (df["Timestamp"] - df["PreviousTimestamp"])
    .dt.total_seconds() / 60
)

df["Is15Min"] = np.isclose(
    df["DiffMinutes"].fillna(-999),
    15.0,
    atol=0.1
)

df["Gap"] = (~df["Is15Min"]).astype(np.int32)

df["Block"] = (
    df.groupby("PatientKey")["Gap"].cumsum()
)

df["BlockKey"] = (
    df["PatientKey"] + "_" + df["Block"].astype(str)
)

block_sizes = df.groupby("BlockKey").size()

valid_blocks = block_sizes[
    block_sizes >= INPUT_STEPS
].index

df = df[df["BlockKey"].isin(valid_blocks)].copy()
df = df.reset_index(drop=True)

df["BlockPosition"] = (
    df.groupby("BlockKey").cumcount()
)

df["CGM_valid"] = (
    df["CGM_obs"] == 1
).astype(np.int8)

df["CGM_8_count"] = (
    df.groupby("BlockKey")["CGM_valid"]
    .rolling(INPUT_STEPS, min_periods=INPUT_STEPS)
    .sum()
    .reset_index(level=0, drop=True)
)

candidate_mask = (
    (df["BlockPosition"] >= INPUT_STEPS - 1)
    & (df["CGM_8_count"] == INPUT_STEPS)
    & df[TARGET_COLUMNS].notna().all(axis=1)
)

candidate_positions = np.flatnonzero(
    candidate_mask.to_numpy()
)

metadata = df.iloc[candidate_positions][
    ["Dataset", "PtID", "Timestamp"]
].reset_index(drop=True)

print("Reconstructed endpoints:", len(metadata))
print("\nDataset distribution:")
print(metadata["Dataset"].value_counts())

print("\nFirst 5 endpoints:")
print(metadata.head())

Reconstructed endpoints: 934004

Dataset distribution:
Dataset
REPLACE-BG       909828
Shanghai-T2DM     24176
Name: count, dtype: int64

First 5 endpoints:
      Dataset  PtID           Timestamp
0  REPLACE-BG    10 1999-12-01 23:00:00
1  REPLACE-BG    10 1999-12-01 23:15:00
2  REPLACE-BG    10 1999-12-01 23:30:00
3  REPLACE-BG    10 1999-12-01 23:45:00
4  REPLACE-BG    10 1999-12-02 00:00:00


In [27]:
import numpy as np

FEATURE_COLUMNS = [
    "CGM", "Carbs", "Carb_events", "Meal_Mass_g", "Meal_Event",
    "Bolus_Insulin", "Basal_Insulin", "CBG", "Blood_Ketone",
    "Hour_sin", "Hour_cos", "Day_sin", "Day_cos",
    "Has_CGM", "Has_Carbs", "Has_Bolus_Insulin",
    "Has_Basal_Insulin", "Has_CBG", "Has_Blood_Ketone",
    "Has_Meal_Mass", "CGM_obs", "Carbs_obs", "Meal_Mass_g_obs",
    "Bolus_Insulin_obs", "Basal_Insulin_obs", "CBG_obs",
    "Blood_Ketone_obs"
]

# Reconstruct targets in original endpoint order
candidate_Y = df.iloc[candidate_positions][
    TARGET_COLUMNS
].to_numpy(dtype=np.float32)

print("Target shape:", candidate_Y.shape)
print("Saved Y shape:", Y_val.shape)
print(
    "Targets match:",
    np.allclose(candidate_Y, Y_val, rtol=0, atol=1e-6)
)

# Compare input windows in batches
total_mismatches = 0
max_difference = 0.0
first_mismatch = None

batch_size = 10000

for start in range(0, len(candidate_positions), batch_size):
    end = min(start + batch_size, len(candidate_positions))
    positions = candidate_positions[start:end]

    offsets = np.arange(7, -1, -1)
    window_indices = positions[:, None] - offsets[None, :]

    batch_X = df.iloc[
        window_indices.ravel()
    ][FEATURE_COLUMNS].to_numpy(
        dtype=np.float32
    ).reshape(-1, 8, 27)

    saved_X = X_val[start:end]

    differences = np.abs(batch_X - saved_X)
    mismatch_mask = np.any(differences > 1e-6, axis=(1, 2))

    total_mismatches += int(mismatch_mask.sum())
    max_difference = max(
        max_difference,
        float(differences.max())
    )

    if first_mismatch is None and mismatch_mask.any():
        local_index = int(np.flatnonzero(mismatch_mask)[0])
        first_mismatch = start + local_index

print("\nINPUT VERIFICATION")
print("Reconstructed X shape:", (len(candidate_positions), 8, 27))
print("Saved X shape:", X_val.shape)
print("Mismatching sequences:", total_mismatches)
print("Maximum absolute difference:", max_difference)
print("First mismatch index:", first_mismatch)
print("Inputs match:", total_mismatches == 0)

Target shape: (934004, 4)
Saved Y shape: (934004, 4)
Targets match: True

INPUT VERIFICATION
Reconstructed X shape: (934004, 8, 27)
Saved X shape: (934004, 8, 27)
Mismatching sequences: 0
Maximum absolute difference: 0.0
First mismatch index: None
Inputs match: True


In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

dataset_labels = metadata["Dataset"].to_numpy()

target_names = ["Target_30", "Target_60", "Target_90", "Target_120"]

for dataset in ["REPLACE-BG", "Shanghai-T2DM"]:

    mask = dataset_labels == dataset

    print("\n" + "=" * 60)
    print("DATASET:", dataset)
    print("Samples:", mask.sum())
    print("=" * 60)

    for i, name in enumerate(target_names):

        mean = norm_params[name]["mean"]
        std = norm_params[name]["std"]

        actual = Y_val[mask, i] * std + mean
        predicted = predictions[mask, i] * std + mean

        mae = mean_absolute_error(actual, predicted)
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        r2 = r2_score(actual, predicted)

        print(f"\n{name} ({(i+1)*30} minutes)")
        print(f"MAE  : {mae:.3f} mg/dL")
        print(f"RMSE : {rmse:.3f} mg/dL")
        print(f"R²   : {r2:.4f}")


DATASET: REPLACE-BG
Samples: 909828

Target_30 (30 minutes)
MAE  : 12.851 mg/dL
RMSE : 18.452 mg/dL
R²   : 0.9187

Target_60 (60 minutes)
MAE  : 23.232 mg/dL
RMSE : 32.349 mg/dL
R²   : 0.7502

Target_90 (90 minutes)
MAE  : 30.994 mg/dL
RMSE : 42.044 mg/dL
R²   : 0.5780

Target_120 (120 minutes)
MAE  : 36.600 mg/dL
RMSE : 48.746 mg/dL
R²   : 0.4327

DATASET: Shanghai-T2DM
Samples: 24176

Target_30 (30 minutes)
MAE  : 9.903 mg/dL
RMSE : 14.366 mg/dL
R²   : 0.9325

Target_60 (60 minutes)
MAE  : 17.247 mg/dL
RMSE : 25.015 mg/dL
R²   : 0.7952

Target_90 (90 minutes)
MAE  : 22.475 mg/dL
RMSE : 32.118 mg/dL
R²   : 0.6621

Target_120 (120 minutes)
MAE  : 26.008 mg/dL
RMSE : 36.753 mg/dL
R²   : 0.5569


In [29]:
from sklearn.metrics import mean_absolute_error
import numpy as np

dataset_labels = metadata["Dataset"].to_numpy()

target_names = [
    "Target_30", "Target_60",
    "Target_90", "Target_120"
]

# Latest observed CGM was already calculated for the baseline.
# Reuse latest_cgm and valid_samples.

for dataset in ["REPLACE-BG", "Shanghai-T2DM"]:

    dataset_mask = (
        (dataset_labels == dataset) & valid_samples
    )

    print("\n" + "=" * 60)
    print("DATASET:", dataset)
    print("Samples:", dataset_mask.sum())
    print("=" * 60)

    for i, name in enumerate(target_names):

        mean = norm_params[name]["mean"]
        std = norm_params[name]["std"]

        actual = Y_val[dataset_mask, i] * std + mean

        baseline = latest_cgm[dataset_mask]

        gpformer = predictions[dataset_mask, i] * std + mean

        persistence_mae = mean_absolute_error(
            actual, baseline
        )

        gpformer_mae = mean_absolute_error(
            actual, gpformer
        )

        improvement = (
            (persistence_mae - gpformer_mae)
            / persistence_mae
        ) * 100

        print(f"\n{name} ({(i+1)*30} minutes)")
        print(f"Persistence MAE: {persistence_mae:.3f}")
        print(f"GPFormer MAE:    {gpformer_mae:.3f}")
        print(f"Improvement:     {improvement:.2f}%")


DATASET: REPLACE-BG
Samples: 909828

Target_30 (30 minutes)
Persistence MAE: 16.696
GPFormer MAE:    12.851
Improvement:     23.03%

Target_60 (60 minutes)
Persistence MAE: 28.546
GPFormer MAE:    23.232
Improvement:     18.62%

Target_90 (90 minutes)
Persistence MAE: 37.419
GPFormer MAE:    30.994
Improvement:     17.17%

Target_120 (120 minutes)
Persistence MAE: 44.212
GPFormer MAE:    36.600
Improvement:     17.22%

DATASET: Shanghai-T2DM
Samples: 24176

Target_30 (30 minutes)
Persistence MAE: 12.566
GPFormer MAE:    9.903
Improvement:     21.19%

Target_60 (60 minutes)
Persistence MAE: 21.175
GPFormer MAE:    17.247
Improvement:     18.55%

Target_90 (90 minutes)
Persistence MAE: 27.803
GPFormer MAE:    22.475
Improvement:     19.16%

Target_120 (120 minutes)
Persistence MAE: 32.871
GPFormer MAE:    26.008
Improvement:     20.88%


In [30]:
from sklearn.metrics import mean_absolute_error
import numpy as np

dataset_labels = metadata["Dataset"].to_numpy()

target_names = [
    "Target_30", "Target_60",
    "Target_90", "Target_120"
]

# Latest observed CGM was already calculated for the baseline.
# Reuse latest_cgm and valid_samples.

for dataset in ["REPLACE-BG", "Shanghai-T2DM"]:

    dataset_mask = (
        (dataset_labels == dataset) & valid_samples
    )

    print("\n" + "=" * 60)
    print("DATASET:", dataset)
    print("Samples:", dataset_mask.sum())
    print("=" * 60)

    for i, name in enumerate(target_names):

        mean = norm_params[name]["mean"]
        std = norm_params[name]["std"]

        actual = Y_val[dataset_mask, i] * std + mean

        baseline = latest_cgm[dataset_mask]

        gpformer = predictions[dataset_mask, i] * std + mean

        persistence_mae = mean_absolute_error(
            actual, baseline
        )

        gpformer_mae = mean_absolute_error(
            actual, gpformer
        )

        improvement = (
            (persistence_mae - gpformer_mae)
            / persistence_mae
        ) * 100

        print(f"\n{name} ({(i+1)*30} minutes)")
        print(f"Persistence MAE: {persistence_mae:.3f}")
        print(f"GPFormer MAE:    {gpformer_mae:.3f}")
        print(f"Improvement:     {improvement:.2f}%")


DATASET: REPLACE-BG
Samples: 909828

Target_30 (30 minutes)
Persistence MAE: 16.696
GPFormer MAE:    12.851
Improvement:     23.03%

Target_60 (60 minutes)
Persistence MAE: 28.546
GPFormer MAE:    23.232
Improvement:     18.62%

Target_90 (90 minutes)
Persistence MAE: 37.419
GPFormer MAE:    30.994
Improvement:     17.17%

Target_120 (120 minutes)
Persistence MAE: 44.212
GPFormer MAE:    36.600
Improvement:     17.22%

DATASET: Shanghai-T2DM
Samples: 24176

Target_30 (30 minutes)
Persistence MAE: 12.566
GPFormer MAE:    9.903
Improvement:     21.19%

Target_60 (60 minutes)
Persistence MAE: 21.175
GPFormer MAE:    17.247
Improvement:     18.55%

Target_90 (90 minutes)
Persistence MAE: 27.803
GPFormer MAE:    22.475
Improvement:     19.16%

Target_120 (120 minutes)
Persistence MAE: 32.871
GPFormer MAE:    26.008
Improvement:     20.88%


In [31]:
from pathlib import Path

base = Path("/content/drive/MyDrive/GLYRA")

required_files = [
    "Development_Train_Sequences.npz",
    "Development_Validation_Sequences.npz",
    "normalization_parameters.json",
    "checkpoints/GPFormer_best.pth",
    "checkpoints/GPFormer_best_backup_epoch8.pth",
    "checkpoints/GPFormer_latest.pth",
]

for file in required_files:
    path = base / file
    print(
        f"{'FOUND' if path.exists() else 'MISSING'}: "
        f"{file}"
    )
    if path.exists():
        print(f"  Size: {path.stat().st_size / (1024**2):.2f} MB")

FOUND: Development_Train_Sequences.npz
  Size: 94.26 MB
FOUND: Development_Validation_Sequences.npz
  Size: 22.99 MB
FOUND: normalization_parameters.json
  Size: 0.00 MB
FOUND: checkpoints/GPFormer_best.pth
  Size: 1.32 MB
FOUND: checkpoints/GPFormer_best_backup_epoch8.pth
  Size: 1.32 MB
FOUND: checkpoints/GPFormer_latest.pth
  Size: 1.32 MB
